# module-composition — ex3: variable-depth MLP via nn.ModuleList

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `module-composition`. Running the final beacon cell reports progress against the `PyTorch: Module composition` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Module composition` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`module-composition`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "module-composition"
DD_SUBTOPIC = "PyTorch: Module composition"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Module composition — quick refresher

Three composition patterns map to three child-registration mechanics:
1. **Named attributes** — `self.fc1 = Linear(...)`. Fixed shape, fully manual.
2. **nn.Sequential** — one container, strictly linear data flow.
3. **nn.ModuleList** — Python-list-flavored container for VARIABLE-DEPTH stacks where you keep manual control over the forward pass (skip connections, gating, conditional routing).

**This drill (ex3) vs prior.** ex1 used named attributes for a fixed two-layer MLP; ex2 used Sequential for the same. ex3 makes depth a constructor argument — only `ModuleList` registers a Python-list-shaped set of children. A plain `self.layers = [...]` list LOOKS like it works but the children would be invisible to `.parameters()` and `.to(device)`.

### Exercise 3 — variable-depth MLP via nn.ModuleList

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `nn.ModuleList` to compose an N-layer MLP whose depth is a constructor argument, then iterate over the list in `forward` while keeping all children visible to `.parameters()`.
> Keywords: nn.ModuleList, variable-depth, list-vs-modulelist, iter-forward
> ```

**KCs targeted:** `modulelist-registers-list-of-children`, `child-module-attribute-registration`

Implement `DeepMLP` — an N-layer MLP whose depth is chosen at construction time.

1. Subclass `t.nn.Module`. `__init__(self, dim, num_layers)`:
   - `super().__init__()` first.
   - `self.layers = t.nn.ModuleList([t.nn.Linear(dim, dim) for _ in range(num_layers)])`. (Plain Python list would NOT register the Linears as children — they'd be invisible to `.parameters()`.)
2. `forward(self, x: Tensor) -> Tensor`:
   - Loop over `self.layers`, applying each in sequence with `t.relu` between (NOT after the last layer).
   - Return the final activation.
3. Return an INSTANCE from `ex3_build_deep_mlp(dim, num_layers)`.

**The test contrasts ModuleList vs plain list.** It also builds a second BROKEN class `BrokenList` that stores the same Linears in a raw `[]` instead of ModuleList, and asserts its `.parameters()` iterator is empty — driving home WHY ModuleList exists.

In [ ]:
def ex3_build_deep_mlp(dim: int, num_layers: int) -> 't.nn.Module':
    """Return an N-layer Linear+ReLU MLP using nn.ModuleList."""
    raise NotImplementedError()


def _test_ex3():
    import torch.nn as nn

    # Build a 3-layer MLP.
    mod = ex3_build_deep_mlp(dim=8, num_layers=3)
    assert isinstance(mod, t.nn.Module)
    assert isinstance(mod.layers, nn.ModuleList), (
        f'mod.layers must be nn.ModuleList, got {type(mod.layers).__name__}'
    )
    assert len(mod.layers) == 3, f'expected 3 layers, got {len(mod.layers)}'
    for layer in mod.layers:
        assert isinstance(layer, nn.Linear)
        assert layer.in_features == 8 and layer.out_features == 8

    # Children visible: ModuleList registers transitively.
    # Each Linear has 2 Parameters (weight, bias) → 3 layers × 2 = 6.
    params = list(mod.parameters())
    assert len(params) == 6, f'expected 6 Parameters (3 layers × 2), got {len(params)}'

    # Forward — preserves shape (dim → dim → dim → dim).
    x = t.randn(2, 8, generator=t.Generator().manual_seed(0))
    y = mod(x)
    assert y.shape == (2, 8), f'output shape {tuple(y.shape)} != (2, 8)'
    assert y.dtype == t.float32

    # Different depth → different param count.
    mod5 = ex3_build_deep_mlp(dim=4, num_layers=5)
    assert len(list(mod5.parameters())) == 10, 'depth-5 should yield 10 Parameters'

    # --- Contrast with the broken plain-list version ---
    class BrokenList(t.nn.Module):
        def __init__(self, dim, num_layers):
            super().__init__()
            self.layers = [t.nn.Linear(dim, dim) for _ in range(num_layers)]
        def forward(self, x):
            for layer in self.layers:
                x = layer(x)
            return x

    broken = BrokenList(dim=8, num_layers=3)
    broken_params = list(broken.parameters())
    assert len(broken_params) == 0, (
        f'plain list should NOT register children — got {len(broken_params)} params. '
        'This is exactly why nn.ModuleList exists.'
    )
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
import torch.nn as nn

class DeepMLP(t.nn.Module):
    def __init__(self, dim, num_layers):
        super().__init__()
        self.layers = nn.ModuleList(
            [nn.Linear(dim, dim) for _ in range(num_layers)]
        )
    def forward(self, x):
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = t.relu(x)
        return x

def ex3_build_deep_mlp(dim, num_layers):
    return DeepMLP(dim, num_layers)
```

**Why ModuleList instead of Sequential?** Sequential locks you into linear data flow — `out = layer3(layer2(layer1(x)))`. ModuleList gives you the SAME registration semantics but you write the forward yourself, which is necessary for skip connections (`x = x + layer(x)`), depth-conditional routing, weight tying, or anything that's not strictly a pipeline.

**The plain-list trap.** Python's `__setattr__` magic for `nn.Module` checks whether the assigned value is a Module / Parameter / Buffer-compatible object. A raw `[Linear(), Linear()]` list passes the isinstance check for `list`, not for any of those registry-eligible types — so the children never get registered. `.parameters()` returns nothing, `.to('cuda')` doesn't move them, `.state_dict()` is empty. **Always use `nn.ModuleList` for list-of-Module patterns.**

**ModuleDict.** The dict analog exists too — `self.heads = nn.ModuleDict({'cls': ..., 'reg': ...})` for name-keyed child Modules.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()